# FedProx μ Retry — Ceftriaxone × *E. coli*

**Re-run only FedProx with a new μ, then update the last run's results in-place.**

Set `TARGET_RUN` and `RETRY_MU` below, then run all cells.

| What's reloaded | What's re-run |
|---|---|
| Data + preprocessing | FedProx with new μ only |
| Centralized MLP/RF baselines | Convergence plot + FedProx per-site plot |
| FedAvg, FedLR, FedRF per-round CSVs | Heatmaps + final_results.csv |
| All best params | Models saved to run's models/ directory |

The target run's existing files are overwritten with updated results.

In [ ]:
# ── CONFIG ──
TARGET_RUN = "03-Run"   # <-- which run to update
RETRY_MU   = 0.1        # <-- new FedProx mu value
NUM_ROUNDS = 30         # use 30 to match the original

In [ ]:
!pip install "flwr[simulation]" maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import warnings, os, io, shutil, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch; import torch.nn as nn; import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
import flwr as fl; import joblib
warnings.filterwarnings("ignore")
SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)
FL_DEVICE = "cpu"; DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Flower: {fl.__version__}  |  Target: {TARGET_RUN}  |  mu: {RETRY_MU}")

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"; DRUG_CSV = "Ceftriaxone"; SPECIES = "Escherichia coli"
PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06-Ceftazidime-E-coli"
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"; MODEL_DIR = RUN_DIR / "models"
OUT_DIR.mkdir(exist_ok=True); MODEL_DIR.mkdir(exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Run dir: {RUN_DIR}")
print(f"Drug: {DRUG_NAME}")

In [ ]:
params_path = OUT_DIR / "best_params_used.txt"
if not params_path.exists(): raise FileNotFoundError(f"{params_path} not found")
params = {}
with open(params_path) as f:
    for line in f:
        if "=" in line: k, v = line.strip().split("=", 1); params[k] = v
BEST_MLP_LR = float(params["BEST_MLP_LR"]); BEST_MLP_DH = float(params["BEST_MLP_DH"])
BEST_MLP_THRESH = float(params["BEST_MLP_THRESH"]); BEST_RF_THRESH = float(params["BEST_RF_THRESH"])
RF_PARAMS = eval(params["RF_PARAMS"])
print(f"MLP: lr={BEST_MLP_LR:.2e}  dropout={BEST_MLP_DH:.1f}  thresh={BEST_MLP_THRESH:.3f}")
print(f"RF:  {RF_PARAMS}  thresh={BEST_RF_THRESH:.3f}")

In [ ]:
THRESHOLDS = np.linspace(0.05, 0.95, 91)
BATCH_SIZE = 16
def n_mixes(n_train):
    if n_train < 1500: return 3
    if n_train < 5000: return 2
    return 1

In [ ]:
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    df_eco = df[df["species"] == SPECIES].copy()
    bin_cols = [c for c in df_eco.columns if c.startswith("bin_")]
    X = df_eco[bin_cols].to_numpy(dtype="float32"); y = df_eco["label"].to_numpy(dtype="int64")
    raw_data[site] = (X, y)
    nr, ns = (y==1).sum(), (y==0).sum()
    print(f"  Site {site}: {len(y)} samples ({ns} S, {nr} R)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")

In [ ]:
client_train, client_test = {}, {}
site_seeds = {"A":42,"B":123,"C":456,"D":789}
for site in SITE_ORDER:
    X, y = raw_data[site]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.10, stratify=y, random_state=site_seeds[site])
    client_train[site] = (X_tr, y_tr); client_test[site] = (X_te, y_te)
    print(f"  Site {site}: train={len(X_tr)} test={len(X_te)}")

In [ ]:
client_train_pp, client_test_pp = {}, {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]; X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
print("Per-site preprocessing done.")

In [ ]:
class DatasetFromNumpy(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def build_mlp():
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256,128),
        dropout_high=BEST_MLP_DH, dropout_low=BEST_MLP_DH/2.0, use_attention=False)

def model_to_numpy(model):
    return [v.cpu().numpy() for v in model.state_dict().values()]

def numpy_to_model(model, params):
    sd = model.state_dict()
    for k, p in zip(sd.keys(), params): sd[k] = torch.tensor(p)
    model.load_state_dict(sd)

def mlp_predict_proba(model, X_np, dev="cpu"):
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(dev)
    with torch.no_grad(): return F.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

In [ ]:
fedavg_hist = []; fedlr_hist = []; fedrf_hist = []; fedprox_histories = {}
centralized_mlp_results = {}; centralized_rf_results = {}
cross_site_results = {}; cross_site_rf_results = {}

if (OUT_DIR / "fedavg_per_round.csv").exists():
    fedavg_hist = pd.read_csv(OUT_DIR / "fedavg_per_round.csv").to_dict("records")
    print(f"Loaded fedavg: {len(fedavg_hist)} rounds")
if (OUT_DIR / "fedlr_per_round.csv").exists():
    fedlr_hist = pd.read_csv(OUT_DIR / "fedlr_per_round.csv").to_dict("records")
    print(f"Loaded fedlr: {len(fedlr_hist)} rounds")
if (OUT_DIR / "fedrf_per_round.csv").exists():
    fedrf_hist = pd.read_csv(OUT_DIR / "fedrf_per_round.csv").to_dict("records")
    print(f"Loaded fedrf: {len(fedrf_hist)} rounds")
for f in sorted(OUT_DIR.glob("fedprox_mu*_per_round.csv")):
    mu_str = f.stem.replace("fedprox_mu","").replace("_per_round","")
    mu = float(mu_str)
    fedprox_histories[mu] = pd.read_csv(f).to_dict("records")
    print(f"Loaded fedprox mu={mu}: {len(fedprox_histories[mu])} rounds")

if (OUT_DIR / "final_results.csv").exists():
    df_old = pd.read_csv(OUT_DIR / "final_results.csv")
    for meth, target in [("Centralized MLP", centralized_mlp_results), ("Centralized RF", centralized_rf_results),
                          ("Cross-Site MLP", cross_site_results), ("Cross-Site RF", cross_site_rf_results)]:
        row = df_old[df_old["Method"] == meth]
        if len(row) > 0:
            r = row.iloc[0]
            for site in SITE_ORDER:
                for suffix in ["BalAcc","AUC"]:
                    col = f"{site}_{suffix}"
                    if col in r and not pd.isna(r[col]): target[col] = float(r[col])
            if "All_BalAcc" in r and not pd.isna(r["All_BalAcc"]): target["All_BalAcc"] = float(r["All_BalAcc"])
            if "All_AUC" in r and not pd.isna(r["All_AUC"]): target["All_AUC"] = float(r["All_AUC"])
    print(f"Loaded centralized + cross-site results")

In [ ]:
def train_local_mixed(model, X_np, y_np, n_mixes, proximal_mu, global_params):
    collect_sds = []; total_loss = 0.0
    for mix_i in range(n_mixes):
        seed = SEED + mix_i * 100 + int(proximal_mu * 1000)
        torch.manual_seed(seed); np.random.seed(seed)
        idx = np.random.permutation(len(X_np))
        X_shuf, y_shuf = X_np[idx], y_np[idx]
        ds = DatasetFromNumpy(X_shuf, y_shuf)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
        opt = torch.optim.AdamW(model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        model.train(); batch_loss = 0.0; n_batch = 0
        for xb, yb in dl:
            if len(xb) <= 1: continue
            xb, yb = xb.to(FL_DEVICE), yb.to(FL_DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            if proximal_mu > 0 and global_params is not None:
                prox = sum((w - gw.to(FL_DEVICE)).norm(2) for w, gw in zip(model.parameters(), global_params))
                loss = loss + (proximal_mu / 2.0) * prox
            loss.backward(); opt.step()
            batch_loss += loss.item(); n_batch += 1
        total_loss += batch_loss / n_batch
        collect_sds.append({k: v.cpu().clone() for k, v in model.state_dict().items()})
    if len(collect_sds) > 1:
        avg_sd = {}
        for k in collect_sds[0].keys():
            stacked = torch.stack([sd[k].float() for sd in collect_sds])
            avg_sd[k] = stacked.mean(0)
            if k.endswith("num_batches_tracked"):
                avg_sd[k] = avg_sd[k].to(torch.long) if avg_sd[k].dim()==0 else avg_sd[k].round().to(torch.long)
        model.load_state_dict(avg_sd)
    return total_loss / n_mixes

class FedMLPClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        self.n_mixes = n_mixes(len(X_train))
        self.model = build_mlp().to(FL_DEVICE)
    def get_parameters(self, config): return model_to_numpy(self.model)
    def set_parameters(self, params): numpy_to_model(self.model, params)
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        proximal_mu = float(config.get("proximal_mu", 0.0))
        global_copy = None
        if proximal_mu > 0: global_copy = [p.clone().detach() for p in self.model.parameters()]
        loss = train_local_mixed(self.model, self.X_train, self.y_train, self.n_mixes, proximal_mu, global_copy)
        return (self.get_parameters({}), len(self.X_train), {"train_loss": loss, "n_mixes": self.n_mixes})

def mlp_client_fn(cid):
    site = SITE_ORDER[int(cid)]
    return FedMLPClient(cid, *client_train_pp[site]).to_client()

In [ ]:
class CheckpointFedProx(fl.server.strategy.FedProx):
    def __init__(self, strategy_name, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.strategy_name = strategy_name
        self.model_dir = Path(model_dir) / strategy_name
        self.model_dir.mkdir(parents=True, exist_ok=True)
    def aggregate_fit(self, server_round, results, failures):
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            m = build_mlp(); numpy_to_model(m, ndarrays)
            torch.save(m.state_dict(), round_dir / f"client_{cp.cid}.pt")
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            gm = build_mlp(); numpy_to_model(gm, nd)
            torch.save(gm.state_dict(), round_dir / "global_model.pt")
        return aggregated, metrics

def get_mlp_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        m = build_mlp(); numpy_to_model(m, parameters); m.to(FL_DEVICE); m.eval()
        record = {"round": server_round}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = mlp_predict_proba(m, X_tt, dev=FL_DEVICE)
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        rd = Path(MODEL_DIR) / f"fedprox_mlp_mu{RETRY_MU}" / f"round_{server_round:03d}"
        if rd.exists():
            with open(rd / "metrics.json", "w") as f: json.dump(record, f)
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

In [ ]:
eval_hist_fedprox = []
_eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_fedprox)

sname = f"fedprox_mlp_mu{RETRY_MU}"
strategy = CheckpointFedProx(sname, MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    proximal_mu=RETRY_MU, evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp())))
strategy.model_dir = Path(MODEL_DIR) / sname

print(f"\n=== FedProx MLP (mu={RETRY_MU}) ===")
fl.simulation.start_simulation(
    client_fn=mlp_client_fn, num_clients=4, config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedprox_histories[RETRY_MU] = eval_hist_fedprox.copy()
print(f"FedProx mu={RETRY_MU} done. {len(eval_hist_fedprox)} rounds.")
last = eval_hist_fedprox[-1] if eval_hist_fedprox else {}
print(f"  Final All_BalAcc: {last.get('All_BalAcc', np.nan):.4f}")
print(f"  Best All_BalAcc:  {max((h.get('All_BalAcc',0) for h in eval_hist_fedprox[1:]), default=0):.4f}")

In [ ]:
def best_metrics(h):
    if not h: return {}, 0
    skip0 = h[1:]
    best = max(skip0, key=lambda r: r.get("All_BalAcc", 0))
    return best, int(best.get("round", 0))

rows = []
def add_row(method, cs_source=None, fed_hist=None):
    r = {"Method": method}; peak_round = ""
    for site in SITE_ORDER:
        if cs_source:
            r[f"{site}_BalAcc"] = cs_source.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = cs_source.get(f"{site}_AUC", np.nan)
        elif fed_hist is not None:
            lm, pr = best_metrics(fed_hist)
            r[f"{site}_BalAcc"] = lm.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = lm.get(f"{site}_AUC", np.nan)
            peak_round = f" (r{pr})"
    if fed_hist is not None:
        lm, pr = best_metrics(fed_hist)
        r["All_BalAcc"] = lm.get("All_BalAcc", np.nan); r["All_AUC"] = lm.get("All_AUC", np.nan)
        r["Peak_Round"] = int(pr)
    elif cs_source:
        r["All_BalAcc"] = cs_source.get("All_BalAcc", np.nan); r["All_AUC"] = cs_source.get("All_AUC", np.nan)
        r["Peak_Round"] = 0
    r["Label"] = method + peak_round; rows.append(r)

add_row("Centralized MLP", cs_source=centralized_mlp_results)
add_row("Centralized RF", cs_source=centralized_rf_results)
if fedavg_hist: add_row("FL FedAvg (MLP)", fed_hist=fedavg_hist)
if RETRY_MU in fedprox_histories:
    add_row(f"FL FedProx mu={RETRY_MU} (MLP)", fed_hist=fedprox_histories[RETRY_MU])
if fedlr_hist: add_row("FL FedAvg (LR)", fed_hist=fedlr_hist)
if fedrf_hist: add_row("FL FedRF (Trees)", fed_hist=fedrf_hist)
if cross_site_results:
    r = {"Method":"Cross-Site MLP","Label":"Cross-Site MLP","Peak_Round":0}
    for site in SITE_ORDER:
        r[f"{site}_BalAcc"] = cross_site_results.get(f"{site}_BalAcc",np.nan)
        r[f"{site}_AUC"] = cross_site_results.get(f"{site}_AUC",np.nan)
    r["All_BalAcc"] = cross_site_results.get("All_BalAcc",np.nan); r["All_AUC"] = cross_site_results.get("All_AUC",np.nan)
    rows.append(r)
if cross_site_rf_results:
    r = {"Method":"Cross-Site RF","Label":"Cross-Site RF","Peak_Round":0}
    for site in SITE_ORDER:
        r[f"{site}_BalAcc"] = cross_site_rf_results.get(f"{site}_BalAcc",np.nan)
        r[f"{site}_AUC"] = cross_site_rf_results.get(f"{site}_AUC",np.nan)
    r["All_BalAcc"] = cross_site_rf_results.get("All_BalAcc",np.nan); r["All_AUC"] = cross_site_rf_results.get("All_AUC",np.nan)
    rows.append(r)

df_results = pd.DataFrame(rows)
cols = ["Method","Label","Peak_Round"] + [f"{s}_BalAcc" for s in SITE_ORDER] + ["All_BalAcc"]
print(df_results[cols].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
if fedavg_hist:
    for site in SITE_ORDER:
        vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedavg_hist]
        ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
    vals_all = [h.get("All_BalAcc", np.nan) for h in fedavg_hist]
    ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
ax.set_title("FedAvg MLP"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

ax = axes[1]
plots_data = []
if fedavg_hist: plots_data.append(("FedAvg MLP", fedavg_hist, "#ff7f0e", "-"))
if RETRY_MU in fedprox_histories:
    plots_data.append((f"FedProx mu={RETRY_MU}", fedprox_histories[RETRY_MU], "#d62728", "--"))
if fedlr_hist: plots_data.append(("FedAvg LR", fedlr_hist, "#1f77b4", "-."))
if fedrf_hist: plots_data.append(("FedRF", fedrf_hist, "#2ca02c", "-"))
for label, hist, c, ls in plots_data:
    vals = [h.get("All_BalAcc", np.nan) for h in hist]
    ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=2, label=label)
    skip0 = hist[1:]
    if skip0:
        best = max(skip0, key=lambda h: h.get("All_BalAcc", 0))
        pr = int(best.get("round", 0))
        ax.axvline(pr, color=c, ls=':', alpha=0.4, lw=1)
        ax.annotate(f"r{pr}", (pr, best.get("All_BalAcc", 0)), textcoords="offset points", xytext=(3,5), fontsize=7, color=c)
if centralized_mlp_results.get("All_BalAcc"):
    ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=2, label='Centralized MLP')
if centralized_rf_results.get("All_BalAcc"):
    ax.axhline(centralized_rf_results["All_BalAcc"], color='gray', ls='--', lw=2, label='Centralized RF')
ax.set_title(f"All Methods (mu={RETRY_MU})"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)
fig.suptitle(f"{DRUG_NAME} x {SPECIES} — FL Convergence", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "convergence.pdf", bbox_inches="tight"); plt.show()

In [ ]:
if RETRY_MU in fedprox_histories:
    fig, ax = plt.subplots(figsize=(8, 5))
    for site in SITE_ORDER:
        vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedprox_histories[RETRY_MU]]
        ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
    vals_all = [h.get("All_BalAcc", np.nan) for h in fedprox_histories[RETRY_MU]]
    ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
    if centralized_mlp_results.get("All_BalAcc"):
        ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=1.5)
    ax.set_title(f"{DRUG_NAME} — FedProx mu={RETRY_MU} Per-Site"); ax.set_xlabel("Round")
    ax.set_ylabel("BalAcc"); ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)
    plt.tight_layout(); plt.savefig(OUT_DIR / "convergence_fedprox.pdf", bbox_inches="tight"); plt.show()

In [ ]:
ba_data = {}
for _, r in df_results.iterrows():
    ba_data[r["Label"]] = {f"Site {s}": r[f"{s}_BalAcc"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_BalAcc", np.nan)): ba_data[r["Label"]]["All"] = r["All_BalAcc"]
df_ba_hm = pd.DataFrame(ba_data).T
df_ba_hm = df_ba_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_ba_hm.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_ba_hm)*0.5)))
sns.heatmap(df_ba_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title(f"{DRUG_NAME} x {SPECIES} — BalAcc (mu={RETRY_MU})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_balacc.pdf", bbox_inches="tight"); plt.show()

In [ ]:
auc_data = {}
for _, r in df_results.iterrows():
    auc_data[r["Label"]] = {f"Site {s}": r[f"{s}_AUC"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_AUC", np.nan)): auc_data[r["Label"]]["All"] = r["All_AUC"]
df_auc_hm = pd.DataFrame(auc_data).T
df_auc_hm = df_auc_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_auc_hm.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_auc_hm)*0.5)))
sns.heatmap(df_auc_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC"}, ax=ax)
ax.set_title(f"{DRUG_NAME} x {SPECIES} — AUC (mu={RETRY_MU})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_auc.pdf", bbox_inches="tight"); plt.show()

In [ ]:
df_results.to_csv(OUT_DIR / "final_results.csv", index=False)
if RETRY_MU in fedprox_histories:
    pd.DataFrame(fedprox_histories[RETRY_MU]).to_csv(
        OUT_DIR / f"fedprox_mu{RETRY_MU}_per_round.csv", index=False)
    print(f"Overwrote fedprox_mu{RETRY_MU}_per_round.csv")

import shutil
src_nb = Path("retry_fedprox.ipynb")
if not src_nb.exists(): src_nb = Path.cwd() / "retry_fedprox.ipynb"
if not src_nb.exists():
    import glob as _g
    candidates = list(_g.glob("/content/**/retry_fedprox.ipynb", recursive=True))
    if candidates: src_nb = Path(candidates[0])
if src_nb.exists(): shutil.copy(str(src_nb), str(OUT_DIR / "retry_notebook.ipynb"))

print(f"\n{'='*60}")
print(f"  Done. Results updated in {OUT_DIR.resolve()}")
for f in sorted(OUT_DIR.glob("*")): print(f"    {f.name}")

---
**Done.** FedProx retry complete.